# 地层与隧道结构 BIM

本文件包含 IFC 辅助函数和可选的点云预处理代码。输入与输出均直接放在本 Notebook 所在目录，不创建子目录，也不依赖外部 Python 脚本。

默认使用同级的 STEP/OFF 地层输入。输入包括 tunnel_axis.json、tunnel_profile.json、Datasource_dikan.xlsx、layer_0.step 至 layer_4.step 和 stratum_1.off 至 stratum_6.off。

依赖：IfcOpenShell、Gmsh、numpy、pandas、openpyxl。可选的体素重建另需 scipy、tqdm 和 pythonocc-core。不生成有限元模型。


In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'diceng2bim_new.ipynb').is_file():
    raise RuntimeError("请从此 Notebook 所在目录启动内核。")
DATA_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR


In [2]:
import json
import math
import gmsh
import sys
import os
import re
import time
import uuid
import pandas as pd
import ifcopenshell
from ifcopenshell import api  # 修复1：严格对齐库文件中的 API 导入方式
import ifcopenshell.util.element

# ==========================================
# 0. 运行模式设置
# ==========================================
PROCESS_MODE = "BOTH"

# ==========================================
# 1. 工程参数与网格设定
# ==========================================
NUM_RINGS = 50         # 向前推进的环数
RING_WIDTH = 1.5       # 真实的 3D 空间弧长

R_GROUT = 3.22         # 注浆层外径
R_SHELL = 3.20         # 衬砌(Lining)外径
R_LINING = 3.1         # 壳体(Shell)外径
R_INNER = 2.75         # 内部空洞半径

R_BOUNDARY = R_GROUT   

# ==========================================
# 2. 基础工具函数
# ==========================================
O = 0., 0., 0.
X = 1., 0., 0.
Y = 0., 1., 0.
Z = 0., 0., 1.

def create_template(filename, timestamp, timestring, creator, organization, application, application_version,
                    project_globalid, project_name):
    template = """ISO-10303-21;
HEADER;
FILE_DESCRIPTION(('ViewDefinition [CoordinationView]'),'2;1');
FILE_NAME('%(filename)s','%(timestring)s',('%(creator)s'),('%(organization)s'),'%(application)s','%(application)s','');
FILE_SCHEMA(('IFC4X3'));
ENDSEC;
DATA;
#1=IFCPERSON($,$,'%(creator)s',$,$,$,$,$);
#2=IFCORGANIZATION($,'%(organization)s',$,$,$);
#3=IFCPERSONANDORGANIZATION(#1,#2,$);
#4=IFCAPPLICATION(#2,'%(application_version)s','%(application)s','');
#5=IFCOWNERHISTORY(#3,#4,$,.ADDED.,$,#3,#4,%(timestamp)s);
#6=IFCDIRECTION((1.,0.,0.));
#7=IFCDIRECTION((0.,0.,1.));
#8=IFCCARTESIANPOINT((0.,0.,0.));
#9=IFCAXIS2PLACEMENT3D(#8,#7,#6);
#10=IFCDIRECTION((0.,1.,0.));
#11=IFCGEOMETRICREPRESENTATIONCONTEXT($,'Model',3,1.E-05,#9,#10);
#12=IFCDIMENSIONALEXPONENTS(0,0,0,0,0,0,0);
#13=IFCSIUNIT(*,.LENGTHUNIT.,$,.METRE.);
#14=IFCSIUNIT(*,.AREAUNIT.,$,.SQUARE_METRE.);
#15=IFCSIUNIT(*,.VOLUMEUNIT.,$,.CUBIC_METRE.);
#16=IFCSIUNIT(*,.PLANEANGLEUNIT.,$,.RADIAN.);
#17=IFCMEASUREWITHUNIT(IFCPLANEANGLEMEASURE(0.017453292519943295),#16);
#18=IFCCONVERSIONBASEDUNIT(#12,.PLANEANGLEUNIT.,'DEGREE',#17);
#19=IFCUNITASSIGNMENT((#13,#14,#15,#18));
#20=IFCPROJECT('%(project_globalid)s',#5,'%(project_name)s',$,$,$,$,(#11),#19);
#21=IFCAXIS2PLACEMENT3D(#8,#7,#6);
#22=IFCGEOMETRICREPRESENTATIONCONTEXT('Model','Model',3,1.E-05,#21);
#23=IFCSIUNIT(.PLANEANGLEUNIT.,.DEGREE.,$);
#24=IFCUNITASSIGNMENT((#13,#14,#15,#23));
ENDSEC;
END-ISO-10303-21;
""" % locals()  
    return template

def create_ifcaxis2placement(ifcfile, point=O, dir1=Z, dir2=X):
    point = ifcfile.createIfcCartesianPoint(point)
    dir1 = ifcfile.createIfcDirection(dir1)
    dir2 = ifcfile.createIfcDirection(dir2)
    axis2placement = ifcfile.createIfcAxis2Placement3D(point, dir1, dir2)
    return axis2placement

def create_ifclocalplacement(ifcfile, point=O, dir1=Z, dir2=X, relative_to=None):
    axis2placement = create_ifcaxis2placement(ifcfile, point, dir1, dir2)
    ifclocalplacement2 = ifcfile.createIfcLocalPlacement(relative_to, axis2placement)
    return ifclocalplacement2

def colour(ifcfile, entities, color, transparency=0.0):
    if color is not None:
        r, g, b = color[0], color[1], color[2]
        owner_history = ifcfile.by_type("IfcOwnerHistory")[0]
        context = ifcfile.by_type("IfcGeometricRepresentationContext")[0]
        rgb = ifcfile.create_entity('IfcColourRgb', "SurfaceColor", float(r), float(g), float(b))
        shading = ifcfile.create_entity('IfcSurfaceStyleShading', rgb, float(transparency))
        style = ifcfile.create_entity('IfcSurfaceStyle', "GeologicalStyle", "BOTH", [shading])
        material = ifcfile.create_entity('IfcMaterial', "GeoMaterial")
        ifcfile.create_entity('IfcRelAssociatesMaterial', ifcopenshell.guid.new(), owner_history, None, None, entities, material)
        styled_item = ifcfile.create_entity('IfcStyledItem', None, [style])
        styled_rep = ifcfile.create_entity('IfcStyledRepresentation', context, "Body", "StyledItem", [styled_item])
        ifcfile.create_entity('IfcMaterialDefinitionRepresentation', None, None, [styled_rep], material)
    else:
        print('Keep original color')

def add_properties(ifcfile, entity, properties, pset_name):
    # 修复2：增强 Pandas 与 Numpy 数据类型的数值清洗，防止底层 C++ 报错跳过绑定
    cleaned_props = {}
    for k, v in properties.items():
        try:
            if pd.isna(v): continue
        except Exception:
            if v is None: continue
            
        if hasattr(v, 'item'): 
            cleaned_props[str(k)] = v.item()
        else: 
            cleaned_props[str(k)] = v

    existing_pset = ifcopenshell.util.element.get_psets(entity).get(pset_name)
    if existing_pset:
        property_set = ifcfile.by_id(existing_pset["id"])
    else:
        property_set = api.run("pset.add_pset", ifcfile, product=entity, name=pset_name)
    api.run("pset.edit_pset", ifcfile, pset=property_set, properties=cleaned_props)

# ==========================================
# 3. 原生几何解析核心 (不变)
# ==========================================
def get_arc_params(x1, y1, x2, y2, bulge):
    dx, dy = x2 - x1, y2 - y1
    L = math.hypot(dx, dy)
    if L < 1e-6 or bulge == 0: return None
    mx, my = x1 + dx / 2.0, y1 + dy / 2.0
    nx, ny = -dy / L, dx / L
    d = (L / 2.0) * (1.0 - bulge**2) / (2.0 * bulge)
    cx, cy = mx + nx * d, my + ny * d
    R = math.hypot(x1 - cx, y1 - cy)
    start_ang = math.atan2(y1 - cy, x1 - cx)
    end_ang = math.atan2(y2 - cy, x2 - cx)
    if bulge > 0:
        sweep_ang = end_ang - start_ang
        while sweep_ang <= 0: sweep_ang += 2 * math.pi
    else:        
        sweep_ang = end_ang - start_ang
        while sweep_ang >= 0: sweep_ang -= 2 * math.pi
    return cx, cy, R, start_ang, sweep_ang

def build_curve_index(points):
    segments = []
    current_dist = 0.0
    for i in range(len(points)-1):
        x1, y1, bulge = points[i]
        x2, y2 = points[i+1][:2]
        if bulge == 0:
            L = math.hypot(x2-x1, y2-y1)
            segments.append({'type': 'line', 'start_dist': current_dist, 'length': L, 'p1': (x1, y1), 'p2': (x2, y2)})
            current_dist += L
        else:
            params = get_arc_params(x1, y1, x2, y2, bulge)
            if params:
                cx, cy, R, start_ang, sweep_ang = params
                L = R * abs(sweep_ang)
                segments.append({'type': 'arc', 'start_dist': current_dist, 'length': L, 'cx': cx, 'cy': cy, 'R': R, 'start_ang': start_ang, 'sweep_ang': sweep_ang})
                current_dist += L
    return segments, current_dist

def get_z_and_slope(profile_data, d):
    if d <= profile_data[0][0]: return profile_data[0][1], 0.0
    if d >= profile_data[-1][0]: return profile_data[-1][1], 0.0
    for i in range(len(profile_data)-1):
        d1, z1 = profile_data[i]
        d2, z2 = profile_data[i+1]
        if d1 <= d <= d2:
            slope = (z2 - z1) / (d2 - d1) if d2 != d1 else 0.0
            return z1 + slope * (d - d1), slope
    return profile_data[-1][1], 0.0

def evaluate_curve_3d(segments, axis_len, profile_data, d):
    pz, slope = get_z_and_slope(profile_data, d)
    d_clipped = max(0.0, min(d, axis_len - 1e-6))
    for seg in segments:
        if seg['start_dist'] <= d_clipped <= seg['start_dist'] + seg['length'] + 1e-6:
            t = (d_clipped - seg['start_dist']) / seg['length']
            if seg['type'] == 'line':
                x1, y1 = seg['p1']; x2, y2 = seg['p2']
                dx, dy = x2 - x1, y2 - y1
                L = math.hypot(dx, dy)
                return x1 + t*dx, y1 + t*dy, pz, dx/L, dy/L, slope
            else:
                cx, cy, R = seg['cx'], seg['cy'], seg['R']
                start_ang, sweep_ang = seg['start_ang'], seg['sweep_ang'] 
                cur_ang = start_ang + t * sweep_ang
                sign = 1 if sweep_ang > 0 else -1
                tx, ty = math.cos(cur_ang + sign*math.pi/2), math.sin(cur_ang + sign*math.pi/2)
                return cx + R*math.cos(cur_ang), cy + R*math.sin(cur_ang), pz, tx, ty, slope
    return None

def get_d_from_3d_length(start_d, target_3d_len, profile_data, max_d):
    current_d = start_d
    remaining_3d = target_3d_len
    if current_d >= max_d: return max_d
    for i in range(len(profile_data)-1):
        d1, z1 = profile_data[i]
        d2, z2 = profile_data[i+1]
        if d2 <= current_d: continue
        seg_start_d = max(current_d, d1)
        seg_end_d = min(d2, max_d)
        if seg_start_d >= seg_end_d: continue
        slope = (z2 - z1) / (d2 - d1) if d2 != d1 else 0.0
        stretch = math.hypot(1.0, slope)
        max_3d_here = (seg_end_d - seg_start_d) * stretch
        if remaining_3d <= max_3d_here + 1e-9: 
            return seg_start_d + remaining_3d / stretch
        else:
            remaining_3d -= max_3d_here
            current_d = seg_end_d
        if current_d >= max_d: break
    if remaining_3d > 0: return current_d + remaining_3d 
    return current_d

# ==========================================
# 4. 核心主干：Gmsh 放样 -> 2D 划分 -> IFC 写入
# ==========================================
def main():
    try:
        with open(str(DATA_DIR / "tunnel_axis.json"), 'r') as f: axis_pts = json.load(f)
        with open(str(DATA_DIR / "tunnel_profile.json"), 'r') as f: profile_raw = json.load(f)
    except Exception as e:
        print(f"未找到 JSON 数据文件，错误信息: {e}")
        return

    excel_path = str(PROJECT_DIR / "Datasource_dikan.xlsx")
    geo_data = []
    tunnel_data = {}
    if os.path.exists(excel_path):
        try:
            df_geo = pd.read_excel(excel_path, sheet_name="Property", index_col=0)
            geo_data = tuple({prop: value for prop, value in df_geo[col].dropna().items()} for col in df_geo.columns)
            df_tunnel = pd.read_excel(excel_path, sheet_name="Tunnel_information")   
            parameters = df_tunnel.iloc[:, 0].tolist() 
            columns = df_tunnel.columns[1:]
            for col in columns:                 
                for i, param in enumerate(parameters):
                    value = df_tunnel.iloc[i, df_tunnel.columns.get_loc(col)]
                    if pd.notna(value):         
                        key = f"{col}_{param}"
                        tunnel_data[key] = value
            print(">>> 成功加载 Excel 勘察与设计语义数据！")
        except Exception as e:
            pass

    y_start = axis_pts[0][1]
    y_end = axis_pts[-1][1]
    if y_start > y_end:
        reversed_pts = []
        for i in range(len(axis_pts)-1, 0, -1):
            reversed_pts.append([axis_pts[i][0], axis_pts[i][1], -axis_pts[i-1][2]])
        reversed_pts.append([axis_pts[0][0], axis_pts[0][1], 0.0])
        axis_pts = reversed_pts

    profile_data = sorted([(item[1], item[0]) for item in profile_raw], key=lambda x: x[0])
    segments, axis_len = build_curve_index(axis_pts)

    start_m = max(0.0, profile_data[0][0])
    end_m = min(axis_len, profile_data[-1][0])
    
    if TARGET_RING > 1:
        total_3d_offset = (TARGET_RING - 1) * RING_WIDTH
        best_m = get_d_from_3d_length(start_m, total_3d_offset, profile_data, end_m)
    else:
        best_m = start_m

    p0 = evaluate_curve_3d(segments, axis_len, profile_data, best_m)
    p0_x, p0_y, p0_z, _, _, _ = p0

    trans_x, trans_y, trans_z = 30.0 - p0_x, 0.0 - p0_y, 0.0          

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 1)
    gmsh.model.add("Tunnel_To_IFC_Model")
    occ = gmsh.model.occ

    def create_wires_at_pt(pt):
        x, y, z, tx, ty, slope = pt
        F_len = math.sqrt(tx**2 + ty**2 + slope**2)
        F = [tx/F_len, ty/F_len, slope/F_len]
        R = [ty, -tx, 0.0]
        loops = {}
        for layer_name, radius in [('grout', R_GROUT), ('shell', R_SHELL), ('lining', R_LINING), ('hole', R_INNER)]:
            circle = occ.addCircle(x, y, z, radius, zAxis=F, xAxis=R)
            wire = occ.addWire([circle])
            loops[layer_name] = wire
        return loops

    v_grouts, v_shells, v_linings, v_inners = [], [], [], []
    current_m = best_m
    
    for i in range(NUM_RINGS):
        m1 = current_m
        m2 = get_d_from_3d_length(m1, RING_WIDTH, profile_data, end_m)
        if m2 >= end_m - 1e-3: break
        pt1 = evaluate_curve_3d(segments, axis_len, profile_data, m1)
        pt2 = evaluate_curve_3d(segments, axis_len, profile_data, m2)
        
        if i == 0:
            x, y, z, tx, ty, slope = pt1
            F_len = math.sqrt(tx**2 + ty**2 + slope**2)
            pt1 = (x - (tx / F_len) * 5.0, y - (ty / F_len) * 5.0, z - (slope / F_len) * 5.0, tx, ty, slope)

        w1 = create_wires_at_pt(pt1)
        w2 = create_wires_at_pt(pt2)
        
        v_g_full = occ.addThruSections([w1['grout'], w2['grout']], makeSolid=True)[0]
        v_s_full = occ.addThruSections([w1['shell'], w2['shell']], makeSolid=True)[0]
        v_l_full = occ.addThruSections([w1['lining'], w2['lining']], makeSolid=True)[0]
        v_h_full = occ.addThruSections([w1['hole'], w2['hole']], makeSolid=True)[0]
        
        v_grouts.append(occ.cut([v_g_full], [v_s_full], removeObject=True, removeTool=False)[0][0])
        v_shells.append(occ.cut([v_s_full], [v_l_full], removeObject=True, removeTool=False)[0][0])
        v_linings.append(occ.cut([v_l_full], [v_h_full], removeObject=True, removeTool=False)[0][0])
        v_inners.append(v_h_full)
        current_m = m2

    tunnel_dimtags = v_grouts + v_shells + v_linings + v_inners
    occ.translate(tunnel_dimtags, trans_x, trans_y, trans_z)

    boundary_box_tag = occ.addBox(0.0, 0.0, -40.0, 60.0, 75.0, 40.0)
    _, intersect_map = occ.intersect(tunnel_dimtags, [(3, boundary_box_tag)], removeObject=True, removeTool=True)
    occ.synchronize()

    all_trimmed_tunnels = [item for sublist in intersect_map for item in sublist]
    
    soil_layer_dimtags = []
    if PROCESS_MODE in ["BOTH", "SOIL_ONLY"]:
        layer_dimtags = []
        base_dir = str(PROJECT_DIR)
        for i in range(5):  
            step_path = os.path.join(base_dir, f"layer_{i}.step")
            if os.path.exists(step_path):
                shapes = occ.importShapes(step_path)
                layer_dimtags.extend(shapes)
        box_tag = occ.addBox(0.0, 0.0, -40.0, 60.0, 75.0, 40.0)
        if layer_dimtags:
            _, soil_layer_map = occ.fragment([(3, box_tag)], layer_dimtags, removeObject=True, removeTool=True)
            occ.synchronize()
            soil_layer_dimtags = [dt for dt in soil_layer_map[0] if dt[0] == 3]
        else:
            soil_layer_dimtags = [(3, box_tag)]
            
    elif PROCESS_MODE == "TUNNEL_ONLY":
        box_tag = occ.addBox(0.0, 0.0, -40.0, 60.0, 75.0, 40.0)
        soil_layer_dimtags = [(3, box_tag)]

    flat_tunnels = []
    tunnel_group_sizes = []
    for sublist in intersect_map:
        flat_tunnels.extend(sublist)
        tunnel_group_sizes.append(len(sublist))

    _, fragment_map = occ.fragment(soil_layer_dimtags, flat_tunnels, removeObject=True, removeTool=True)
    occ.synchronize()

    def filter_valid_volumes(dimtags, label_name):
        valid_list = []
        for dt in dimtags:
            if dt[0] != 3: continue
            try:
                if gmsh.model.occ.getMass(dt[0], dt[1]) > 0.0005: valid_list.append(dt)
            except Exception: pass
        return valid_list

    num_soil_inputs = len(soil_layer_dimtags)
    tunnel_fragment_tags = set(dt[1] for sublist in fragment_map[num_soil_inputs:] for dt in sublist if dt[0] == 3)

    raw_soil_infos = []
    for i in range(num_soil_inputs):
        raw_frags = [dt for dt in fragment_map[i] if dt[0] == 3]
        if not raw_frags: continue
        sum_z = sum(gmsh.model.occ.getCenterOfMass(3, tag)[2] for dim, tag in raw_frags)
        raw_soil_infos.append({'index': i, 'avg_z': sum_z / len(raw_frags), 'frags': raw_frags})
    raw_soil_infos.sort(key=lambda x: x['avg_z'], reverse=True)

    fragment_to_soil_name = {}
    soil_strata_dimtags = [] 
    for rank, info in enumerate(raw_soil_infos):
        layer_name = f"Soil{rank+1}"
        for dim, tag in info['frags']: fragment_to_soil_name[tag] = layer_name
        clean_soil_frags = [dt for dt in info['frags'] if dt[1] not in tunnel_fragment_tags]
        clean_soil_frags = filter_valid_volumes(clean_soil_frags, f"地层 {layer_name}")
        if clean_soil_frags:
            tags = [dt[1] for dt in clean_soil_frags]
            gmsh.model.addPhysicalGroup(3, tags, name=layer_name)
            soil_strata_dimtags.extend(clean_soil_frags)

    trimmed_tunnel_maps = []
    flat_idx = 0
    for size in tunnel_group_sizes:
        sublist_fragments = []
        for _ in range(size):
            sublist_fragments.extend(fragment_map[num_soil_inputs + flat_idx])
            flat_idx += 1
        trimmed_tunnel_maps.append(sublist_fragments)

    def filter_list_of_lists(list_of_lists, label_name):
        return [filter_valid_volumes(sublist, label_name) for sublist in list_of_lists]

    new_v_grouts = filter_list_of_lists(trimmed_tunnel_maps[0:NUM_RINGS], "注浆层")
    new_v_shells = filter_list_of_lists(trimmed_tunnel_maps[NUM_RINGS:2*NUM_RINGS], "外壳")
    new_v_linings = filter_list_of_lists(trimmed_tunnel_maps[2*NUM_RINGS:3*NUM_RINGS], "衬砌")
    new_v_inners = filter_list_of_lists(trimmed_tunnel_maps[3*NUM_RINGS:4*NUM_RINGS], "内洞")
    all_trimmed_tunnels = [item for sublist in (new_v_grouts + new_v_shells + new_v_linings + new_v_inners) for item in sublist]

    if PROCESS_MODE in ["BOTH", "TUNNEL_ONLY"]:
        tunnel_groups = {}
        for i in range(NUM_RINGS):
            actual_ring_num = TARGET_RING + i
            layer_mappings = [
                (new_v_inners, "Excavation"), (new_v_linings, "Lining"),
                (new_v_shells, "Shell"), (new_v_grouts, "Grout")  
            ]
            for layer_list, base_name in layer_mappings:
                for dim, tag in layer_list[i]:
                    soil_name = fragment_to_soil_name.get(tag, "")
                    group_name = f"{soil_name}_{base_name}{actual_ring_num}" if soil_name else f"{base_name}{actual_ring_num}"
                    if group_name not in tunnel_groups: tunnel_groups[group_name] = []
                    tunnel_groups[group_name].append(tag)
        for g_name, tags in tunnel_groups.items():
            gmsh.model.addPhysicalGroup(3, tags, name=g_name)

    # ---------------- 2D 封闭表面网格 ----------------
    print("正在生成用于 IFC 表达的精细 2D 封闭表面网格...")
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.MeshSizeMin", 0.6)
    gmsh.option.setNumber("Mesh.MeshSizeMax", 6.0) 
    
    tunnel_surfaces = set(b[1] for dim, tag in all_trimmed_tunnels for b in gmsh.model.getBoundary([(3, tag)]))
    strata_surfaces = set(b_tag for dim, tag in soil_strata_dimtags for b_dim, b_tag in gmsh.model.getBoundary([(3, tag)]) if b_tag not in tunnel_surfaces)
                
    field_list = []
    if tunnel_surfaces:
        gmsh.model.mesh.field.add("Distance", 1)
        gmsh.model.mesh.field.setNumbers(1, "SurfacesList", list(tunnel_surfaces))
        gmsh.model.mesh.field.add("Threshold", 2)
        gmsh.model.mesh.field.setNumber(2, "InField", 1)    
        gmsh.model.mesh.field.setNumber(2, "SizeMin", 0.6)  
        gmsh.model.mesh.field.setNumber(2, "SizeMax", 6.0)  
        gmsh.model.mesh.field.setNumber(2, "DistMin", 1.0)  
        gmsh.model.mesh.field.setNumber(2, "DistMax", 6.0) 
        field_list.append(2)
    
    if strata_surfaces:
        gmsh.model.mesh.field.add("Distance", 3)
        gmsh.model.mesh.field.setNumbers(3, "SurfacesList", list(strata_surfaces))
        gmsh.model.mesh.field.add("Threshold", 4)
        gmsh.model.mesh.field.setNumber(4, "InField", 3)    
        gmsh.model.mesh.field.setNumber(4, "SizeMin", 2.0)  
        gmsh.model.mesh.field.setNumber(4, "SizeMax", 6.0)  
        gmsh.model.mesh.field.setNumber(4, "DistMin", 0.5)  
        gmsh.model.mesh.field.setNumber(4, "DistMax", 1.0) 
        field_list.append(4)
    
    if len(field_list) == 2:
        gmsh.model.mesh.field.add("Min", 5)
        gmsh.model.mesh.field.setNumbers(5, "FieldsList", field_list)
        gmsh.model.mesh.field.setAsBackgroundMesh(5)
    elif len(field_list) == 1:
        gmsh.model.mesh.field.setAsBackgroundMesh(field_list[0])
    
    gmsh.option.setNumber("Mesh.Algorithm", 6)
    gmsh.option.setNumber("Mesh.Optimize", 1)
    gmsh.model.mesh.generate(2)

    # ==========================================
    # 5. 回归本质：通过模板生成结构完美的 IFC 文件
    # ==========================================
    print(f"正在内存中构建紧密绑定的 IFC 4.3 实体关系 ({PROCESS_MODE})...")
    create_guid = lambda: ifcopenshell.guid.compress(uuid.uuid1().hex)
    ifc_filename = str(OUTPUT_DIR / f"tunnel_model_ring{TARGET_RING}_{PROCESS_MODE}.ifc")
    
    node_tags, coords, _ = gmsh.model.mesh.getNodes()
    global_nodes = {tag: (float(coords[i*3]), float(coords[i*3+1]), float(coords[i*3+2])) for i, tag in enumerate(node_tags)}

    timestamp = int(time.time())
    timestring = time.strftime("%Y-%m-%dT%H:%M:%S", time.gmtime(timestamp))
    
    safe_filename = "tunnel_model.ifc"
    safe_project_name = "Tunnel_Project"
    
    template = create_template(
        filename=safe_filename, 
        timestamp=timestamp, 
        timestring=timestring, 
        creator="Shu Jiang", 
        organization="WHU", 
        application="IfcOpenShell", 
        application_version="0.7", 
        project_globalid=create_guid(), 
        project_name=safe_project_name
    )
    

    model = ifcopenshell.file.from_string(template)  # 在内存中解析 IFC 模板
    
    owner_history = model.by_type("IfcOwnerHistory")[0]
    project = model.by_type("IfcProject")[0]
    context = model.by_type("IfcGeometricRepresentationContext")[0]
    
    project.Name = f"隧道施工BIM(起始环{TARGET_RING})"
    
    # 修复3：构建标准的 IFC 空间层级关系 (Project -> Site -> Building -> Storey)，绝大多数查看器强依赖此树形结构来展示信息
    site_placement = create_ifclocalplacement(model)
    site = model.createIfcSite(create_guid(), owner_history, "Site", None, None, site_placement, None, None, "ELEMENT", None, None, None, None, None)
    
    building_placement = create_ifclocalplacement(model, relative_to=site_placement)
    building = model.createIfcBuilding(create_guid(), owner_history, 'Building', None, None, building_placement, None, None, "ELEMENT", None, None, None)
    
    storey_placement = create_ifclocalplacement(model, relative_to=building_placement)
    building_storey = model.createIfcBuildingStorey(create_guid(), owner_history, 'Storey', None, None, storey_placement, None, None, "ELEMENT", 0.0)

    model.createIfcRelAggregates(create_guid(), owner_history, "Project Container", None, project, [site])
    model.createIfcRelAggregates(create_guid(), owner_history, "Site Container", None, site, [building])
    model.createIfcRelAggregates(create_guid(), owner_history, "Building Container", None, building, [building_storey])


    for dim, p_tag in gmsh.model.getPhysicalGroups(3):
        group_name = gmsh.model.getPhysicalName(dim, p_tag)
        v_tags = gmsh.model.getEntitiesForPhysicalGroup(dim, p_tag)
        
        surf_count = {}
        for v_tag in v_tags:
            for b_dim, s_tag in gmsh.model.getBoundary([(3, v_tag)]):
                surf_count[s_tag] = surf_count.get(s_tag, 0) + 1
                
        outer_surfaces = [s_tag for s_tag, count in surf_count.items() if count == 1]
        v_triangles = []
        for s in outer_surfaces:
            try:
                _, _, elem_node_tags = gmsh.model.mesh.getElements(2, s)
                for item in elem_node_tags:
                    for idx in range(0, len(item), 3):
                        v_triangles.append((item[idx], item[idx+1], item[idx+2]))
            except Exception: pass
        if not v_triangles: continue

        unique_tags = list(set([tag for tri in v_triangles for tag in tri]))
        local_map = {tag: idx + 1 for idx, tag in enumerate(unique_tags)}
        vertices = [global_nodes[tag] for tag in unique_tags]
        faces = [(int(local_map[tri[0]]), int(local_map[tri[1]]), int(local_map[tri[2]])) for tri in v_triangles]

        space_placement = create_ifclocalplacement(model, relative_to=storey_placement)
        
        # 修复4：弃用兼容性极差的 IfcGeotechnicalStratum，改用通用且具有最广支持度的 IfcBuildingElementProxy（类似于示范文件中的 IfcSpace/IfcAirTerminalBox）
        stratum = model.create_entity('IfcBuildingElementProxy', create_guid())
        stratum.OwnerHistory = owner_history
        stratum.Name = group_name
        stratum.ObjectPlacement = space_placement

        point_list = model.create_entity('IfcCartesianPointList3D', vertices)
        polygon_faces = [model.create_entity('IfcIndexedPolygonalFace', face) for face in faces]
        polygon_faces_set = model.create_entity('IfcPolygonalFaceSet', point_list, True, polygon_faces)
        shape_rep = model.create_entity("IfcShapeRepresentation", context, "Body", "Tessellation", [polygon_faces_set])
        stratum.Representation = model.create_entity("IfcProductDefinitionShape", None, None, [shape_rep])

        # 修复5：将独立元素利用 RelContainedInSpatialStructure 包裹到标准结构内，而不是 Aggregate 它
        model.create_entity('IfcRelContainedInSpatialStructure', create_guid(), owner_history, None, None, [stratum], building_storey)

        semantic_props = {}
        color_val, transparency_val = [0.8, 0.8, 0.8], 0.05

        if "Lining" in group_name:
            color_val = [0.7, 0.7, 0.7]
            semantic_props = {k: v for k, v in tunnel_data.items() if "Lining segment" in k} or {"Material": "Concrete", "Young": 25000}
        elif "Shell" in group_name:
            color_val = [0.4, 0.4, 0.4]
            semantic_props = {k: v for k, v in tunnel_data.items() if "Shield shell" in k} or {"Material": "Steel", "Young": 206000}
        elif "Grout" in group_name:
            color_val = [0.65, 0.58, 0.46]
            semantic_props = {k: v for k, v in tunnel_data.items() if "grouting layer" in k} or {"Material": "Cement Grout", "Young": 400}
        elif "Excavation" in group_name:
            color_val, transparency_val = [1.0, 1.0, 1.0], 0.95
            semantic_props = {"Material": "Air", "Density": 1.225}
        elif "Soil" in group_name:
            match = re.search(r'Soil(\d+)', group_name)
            if match:
                idx = int(match.group(1))
                soil_colors = [[1, 0.973, 0.863], [0, 0.749, 1], [0.545, 0.451, 0.333], [1, 0.843, 0], [0, 0.604, 0.804], [0.063, 0.306, 0.545]]
                color_val = soil_colors[(idx - 1) % len(soil_colors)]
                if geo_data and idx <= len(geo_data): semantic_props = geo_data[len(geo_data) - idx]
            if not semantic_props: semantic_props = {"Material": "Soil", "model": "HSS"}

        if semantic_props:
            add_properties(model, stratum, semantic_props, "CustomPropertySet")
        
        colour(model, [stratum], color_val, transparency_val)

    model.write(ifc_filename)
    print(f">>> 大功告成！直通输出！模型已保存至 -> {ifc_filename}\n")
    gmsh.finalize()

if __name__ == "__main__":
    TARGET_RING = 425
    main()

>>> 成功加载 Excel 勘察与设计语义数据！
正在生成用于 IFC 表达的精细 2D 封闭表面网格...
正在内存中构建紧密绑定的 IFC 4.3 实体关系 (BOTH)...
>>> 大功告成！直通输出！模型已保存至 -> E:\Github\geo_bim_model\tunnel_model_ring425_BOTH.ifc

